# ViInfographicVQA — OCR-augmented prompting (single-image task)

Hypothesis: the model sees each infographic downscaled to about one megapixel, so small text is lost.
Running OCR on a larger version of the image and pasting the text into the prompt should recover it.

This notebook is `01_baseline_zero_shot.ipynb` with one change: the prompt also carries OCR text.
Run notebook 01 first with the same `MODEL_ID` — the last cell compares against its results.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
OCR output and answers are cached on Drive, so re-running resumes where it stopped.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LIMIT = None            # e.g. 50 for a quick trial run; None = the whole test split
SAVE_TO_DRIVE = True
LOAD_IN_4BIT = False    # set True for the 7B model on a 16 GB GPU

MODE = "image+ocr"      # "image+ocr" = image and OCR text; "ocr_only" = OCR text without the image (ablation)
OCR_CANVAS = 2560       # longest image side given to the OCR engine, in pixels
OCR_MIN_CONF = 0.3      # drop OCR boxes below this confidence
MAX_OCR_CHARS = 3000    # truncate very long OCR text

# Same image budget as the paper's Qwen wrapper.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28
MAX_NEW_TOKENS = 100

DATASET = "duytranus/ViInfographicVQA"
SPLIT_FILE = "data/single_test.json"

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate qwen-vl-utils Levenshtein bitsandbytes easyocr

In [ ]:
import json, os
from pathlib import Path

assert MODE in ("image+ocr", "ocr_only")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/viinfographicvqa")
else:
    ROOT = Path("/content")
OUT_DIR = ROOT / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = MODEL_ID.split("/")[-1]
OUT_FILE = OUT_DIR / f"single_test__{MODEL_NAME}__{MODE.replace('+', '-')}.json"
BASELINE_FILE = OUT_DIR / f"single_test__{MODEL_NAME}.json"
OCR_FILE = ROOT / "ocr" / f"easyocr_canvas{OCR_CANVAS}.json"
OCR_FILE.parent.mkdir(parents=True, exist_ok=True)
print("results ->", OUT_FILE)

## Data: annotations plus only the images the test split needs (~250 MB)

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

DATA_DIR = Path("/content/ViInfographicVQA")
hf_hub_download(DATASET, SPLIT_FILE, repo_type="dataset", local_dir=DATA_DIR)
rows = json.loads((DATA_DIR / SPLIT_FILE).read_text(encoding="utf-8"))
if LIMIT:
    rows = rows[:LIMIT]
images = sorted({r["image_path"] for r in rows})
snapshot_download(DATASET, repo_type="dataset", local_dir=DATA_DIR, allow_patterns=images)
missing = [p for p in images if not (DATA_DIR / p).exists()]
assert not missing, f"{len(missing)} images failed to download, e.g. {missing[:3]}"
print(len(rows), "questions on", len(images), "images")

## OCR (once per image, cached)

EasyOCR with the Vietnamese model. Boxes are grouped into lines and read left to right, top to bottom.
Expect some wrong diacritics and digits; the prompt tells the model the text may contain errors.

In [ ]:
import easyocr, torch
from tqdm.auto import tqdm


def reading_order(dets, line_tol=0.6):
    """Group OCR boxes into lines by vertical position, then read each line left to right."""
    items = []
    for box, text, conf in dets:
        xs, ys = [p[0] for p in box], [p[1] for p in box]
        items.append({"x": min(xs), "cy": (min(ys) + max(ys)) / 2, "h": max(ys) - min(ys), "text": text})
    items.sort(key=lambda d: d["cy"])
    lines = []
    for it in items:
        if lines and abs(it["cy"] - lines[-1][-1]["cy"]) < line_tol * max(it["h"], lines[-1][-1]["h"]):
            lines[-1].append(it)
        else:
            lines.append([it])
    return "\n".join(" ".join(d["text"] for d in sorted(line, key=lambda d: d["x"])) for line in lines)


ocr = json.loads(OCR_FILE.read_text(encoding="utf-8")) if OCR_FILE.exists() else {}
todo = [p for p in images if p not in ocr]
if todo:
    reader = easyocr.Reader(["vi", "en"], gpu=torch.cuda.is_available(), verbose=False)
    for i, p in enumerate(tqdm(todo, desc="OCR")):
        dets = reader.readtext(str(DATA_DIR / p), canvas_size=OCR_CANVAS, min_size=10)
        ocr[p] = reading_order([d for d in dets if d[2] >= OCR_MIN_CONF])
        if i % 10 == 0:
            OCR_FILE.write_text(json.dumps(ocr, ensure_ascii=False), encoding="utf-8")
    OCR_FILE.write_text(json.dumps(ocr, ensure_ascii=False), encoding="utf-8")
    del reader
    torch.cuda.empty_cache()

lengths = sorted(len(ocr[p]) for p in images)
print(f"OCR text per image: median {lengths[len(lengths) // 2]} chars, max {lengths[-1]}, "
      f"{sum(n > MAX_OCR_CHARS for n in lengths)} images will be truncated, {sum(n == 0 for n in lengths)} empty")
print("--- sample:", images[0])
print(ocr[images[0]][:600])

## Model

In [ ]:
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
kwargs = dict(torch_dtype=torch.bfloat16, device_map="cuda")
if LOAD_IN_4BIT:
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, use_fast=True)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 1e9:.1f} GB used")

## Inference

Identical to notebook 01 (same system prompt, greedy decoding) except for the OCR block in the user message.

In [ ]:
SYSTEM_PROMPT = ("Answer the following question based solely on the image content "
                 "concisely with a single term.")
OCR_HEADER = "Text extracted from the image by OCR (it may contain errors; trust the image when they disagree):"


def build_messages(question: str, image_path: str, ocr_text: str) -> list:
    text = f"{OCR_HEADER}\n{ocr_text[:MAX_OCR_CHARS]}\n\nQuestion: {question}\nAnswer:"
    content = [{"type": "text", "text": text}]
    if MODE == "image+ocr":
        content.insert(0, {"type": "image", "image": f"file://{image_path}"})
    return [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": content}]


def parse_answer(response: str) -> str:
    return response.split("Answer:")[-1].strip() if "Answer:" in response else response.strip()


@torch.no_grad()
def infer(question: str, image_path: str, ocr_text: str) -> str:
    messages = build_messages(question, image_path, ocr_text)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = out[:, inputs.input_ids.shape[1]:]
    return parse_answer(processor.batch_decode(new_tokens, skip_special_tokens=True)[0])


done = {}
if OUT_FILE.exists():
    done = {r["question_id"]: r["predict"] for r in json.loads(OUT_FILE.read_text(encoding="utf-8"))
            if "predict" in r}
    print("resuming:", len(done), "answers already saved")


def save():
    OUT_FILE.write_text(json.dumps(rows, ensure_ascii=False, indent=1), encoding="utf-8")


for i, row in enumerate(tqdm(rows)):
    if row["question_id"] in done:
        row["predict"] = done[row["question_id"]]
        continue
    row["predict"] = infer(row["question"], str(DATA_DIR / row["image_path"]), ocr[row["image_path"]])
    if i % 25 == 0:
        save()
save()
print("saved", OUT_FILE)

## Score

`ANLS` reproduces the paper's scorer exactly; `ANLS-std` is the textbook definition; `Acc` is exact match.
Paper, single-image overall ANLS: Qwen2.5-VL-7B 67.42 · fine-tuned 67.80 · InternVL3.5-8B 67.02 · Ovis2.5-9B 71.02.

In [ ]:
from collections import defaultdict

import Levenshtein

THRESHOLD = 0.5


def clean_prediction(text: str) -> str:
    return text.lower().strip().rstrip(".").replace('"', "").rstrip(">").lstrip("<")


def clean_answer(text: str) -> str:
    return text.lower().strip()


def paper_anls(answer: str, prediction: str) -> float:
    p = clean_prediction(prediction).replace('"', "").rstrip(".").lower()
    score = Levenshtein.ratio(p, clean_answer(answer))
    return score if score >= THRESHOLD else 0.0


def standard_anls(answer: str, prediction: str) -> float:
    p, g = clean_prediction(prediction), clean_answer(answer)
    if not p and not g:
        return 1.0
    nl = Levenshtein.distance(p, g) / max(len(p), len(g))
    return 1.0 - nl if nl < THRESHOLD else 0.0


def exact_match(answer: str, prediction: str) -> float:
    return float(clean_prediction(prediction) == clean_answer(answer))


METRICS = {"anls": paper_anls, "anls_std": standard_anls, "acc": exact_match}

# Fields to break results down by; whichever exist in the split are used.
GROUP_FIELDS = ("answer_source", "element", "image_type")


def _as_labels(value) -> list[str]:
    """`element` is a string in most rows but a list in a few."""
    if value is None:
        return []
    return [str(v) for v in value] if isinstance(value, list) else [str(value)]


def score(rows: list[dict], pred_key: str = "predict") -> dict:
    """Return {"overall": {...}, "<field>": {"<label>": {...}}} with scores in percent."""
    sums: dict = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    for row in rows:
        vals = {name: fn(row["answer"], str(row.get(pred_key, ""))) for name, fn in METRICS.items()}
        groups = [("overall", "overall")]
        groups += [(f, label) for f in GROUP_FIELDS for label in _as_labels(row.get(f))]
        for field, label in groups:
            bucket = sums[field][label]
            bucket["count"] += 1
            for name, v in vals.items():
                bucket[name] += v
    out: dict = {}
    for field, labels in sums.items():
        table = {label: {"count": int(b["count"]),
                         **{m: round(100 * b[m] / b["count"], 2) for m in METRICS}}
                 for label, b in labels.items()}
        out[field] = table["overall"] if field == "overall" else table
    return out


def format_report(result: dict) -> str:
    lines = [f"{'':32s}{'n':>6s}{'ANLS':>8s}{'ANLS-std':>10s}{'Acc':>8s}"]

    def row(name: str, s: dict) -> str:
        return f"{name[:31]:32s}{s['count']:6d}{s['anls']:8.2f}{s['anls_std']:10.2f}{s['acc']:8.2f}"

    lines.append(row("Overall", result["overall"]))
    for field in GROUP_FIELDS:
        if field in result:
            lines.append(f"-- {field}")
            for label, s in sorted(result[field].items(), key=lambda kv: -kv[1]["count"]):
                lines.append(row("  " + label, s))
    return "\n".join(lines)


print(MODEL_ID, "|", MODE)
result = score(rows)
print(format_report(result))

## Compare with the baseline from notebook 01 (same model, no OCR)

In [ ]:
if not BASELINE_FILE.exists():
    print("No baseline file at", BASELINE_FILE, "- run notebook 01 with the same MODEL_ID first.")
else:
    base_pred = {r["question_id"]: r["predict"] for r in json.loads(BASELINE_FILE.read_text(encoding="utf-8"))
                 if "predict" in r}
    both = [r for r in rows if r["question_id"] in base_pred]
    base_rows = [{**r, "predict": base_pred[r["question_id"]]} for r in both]
    base, ours = score(base_rows), score(both)
    print(f"{len(both)} questions answered by both runs\n")
    print(f"{'':32s}{'n':>6s}{'baseline':>10s}{'+OCR':>8s}{'delta':>8s}")

    def line(name, b, o):
        print(f"{name[:31]:32s}{o['count']:6d}{b['anls']:10.2f}{o['anls']:8.2f}{o['anls'] - b['anls']:+8.2f}")

    line("Overall (ANLS)", base["overall"], ours["overall"])
    for field in ("answer_source", "element"):
        print("--", field)
        for label in sorted(ours[field], key=lambda k: -ours[field][k]["count"]):
            line("  " + label, base[field][label], ours[field][label])

    fixed = [r for r, b in zip(both, base_rows)
             if paper_anls(r["answer"], b["predict"]) == 0 and paper_anls(r["answer"], r["predict"]) > 0]
    broken = [r for r, b in zip(both, base_rows)
              if paper_anls(r["answer"], b["predict"]) > 0 and paper_anls(r["answer"], r["predict"]) == 0]
    print(f"\nwrong -> right with OCR: {len(fixed)}   right -> wrong with OCR: {len(broken)}")
    for title, group in (("FIXED by OCR", fixed), ("BROKEN by OCR", broken)):
        print(f"\n== {title} (first 8)")
        for r in group[:8]:
            print(f"[{r['answer_source']} | {r['element']}] {r['question']}\n   gold: {r['answer']}\n"
                  f"   base: {base_pred[r['question_id']]}\n   +ocr: {r['predict']}\n")